In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import subprocess
import json
from typing import Dict
import os
import cv2
import numpy as np

In [ ]:
import sys
sys.path.append("../src/")

In [ ]:
from drone_core import ecef_from_gps

## Helper fucntions

In [ ]:
def read_video_metadata_exiftool(video_path:str) -> dict:
    cmd = ["exiftool", "-ee", "-u", "-j", "-G3", "-a", "-n", video_path]
    out = subprocess.check_output(cmd, text=True)
    return json.loads(out)[0]

In [ ]:
def write_gimble_tags(metadata, image_file):
    cmd = ["exiftool", "-overwrite_original",
           f"-XMP-drone-dji:GimbalYawDegree={metadata['XMP-drone-dji:GimbalYawDegree']}",
           f"-XMP-drone-dji:GimbalPitchDegree={metadata['XMP-drone-dji:GimbalPitchDegree']}",
           f"-XMP-drone-dji:GimbalRollDegree={metadata['XMP-drone-dji:GimbalRollDegree']}",
           f"{image_file}"]
    result = subprocess.run(cmd, capture_output=True, text=True)
    return result

In [ ]:
def write_gps_tags(metadata, image_file):
    cmd = ["exiftool", "-overwrite_original",
           f"-XMP-drone-dji:GPSLatitude={metadata['XMP-drone-dji:GPSLatitude']}",
           f"-XMP-drone-dji:GPSLongitude={metadata['XMP-drone-dji:GPSLongitude']}",
           f"-XMP-drone-dji:AbsoluteAltitude={metadata['XMP-drone-dji:AbsoluteAltitude']}",
           f"-XMP-drone-dji:RelativeAltitude={metadata['XMP-drone-dji:RelativeAltitude']}",
           f"{image_file}"]
    result = subprocess.run(cmd, capture_output=True, text=True)
    return result

In [ ]:
def write_standard_gps_tags(metadata, image_file):
    latitudeRef = 'N' if metadata['XMP-drone-dji:GPSLatitude'] > 0 else 'S'
    longitudeRef = 'E' if metadata['XMP-drone-dji:GPSLongitude'] > 0 else 'W'
    altitudeRef = 0 if metadata['XMP-drone-dji:AbsoluteAltitude'] > 0 else 1

    cmd = ["exiftool", "-overwrite_original",
           f"-GPS:GPSLatitudeRef={latitudeRef}",
           f"-GPS:GPSLatitude={abs(metadata['XMP-drone-dji:GPSLatitude'])}",
           f"-GPS:GPSLongitudeRef={longitudeRef}",
           f"-GPS:GPSLongitude={abs(metadata['XMP-drone-dji:GPSLongitude'])}",
           f"-GPS:GPSAltitudeRef={altitudeRef}",
           f"-GPS:GPSAltitude={abs(metadata['XMP-drone-dji:AbsoluteAltitude'])}",
           f"{image_file}"]
    result = subprocess.run(cmd, capture_output=True, text=True)
    return result

In [ ]:
def write_drone_tags(metadata, image_file):
    cmd = ["exiftool", "-overwrite_original",
           f"-XMP-drone-dji:FlightYawDegree={metadata['XMP-drone-dji:FlightYawDegree']}",
           f"-XMP-drone-dji:FlightPitchDegree={metadata['XMP-drone-dji:FlightPitchDegree']}",
           f"-XMP-drone-dji:FlightRollDegree={metadata['XMP-drone-dji:FlightRollDegree']}",
           f"{image_file}"]
    result = subprocess.run(cmd, capture_output=True, text=True)
    return result

In [ ]:
def write_all_drone_tags(metadata, image_file):
    cmd = ["exiftool", "-overwrite_original",
           f"-XMP-drone-dji:GimbalYawDegree={metadata['XMP-drone-dji:GimbalYawDegree']}",
           f"-XMP-drone-dji:GimbalPitchDegree={metadata['XMP-drone-dji:GimbalPitchDegree']}",
           f"-XMP-drone-dji:GimbalRollDegree={metadata['XMP-drone-dji:GimbalRollDegree']}",
           f"-XMP-drone-dji:GPSLatitude={metadata['XMP-drone-dji:GPSLatitude']}",
           f"-XMP-drone-dji:GPSLongitude={metadata['XMP-drone-dji:GPSLongitude']}",
           f"-XMP-drone-dji:AbsoluteAltitude={metadata['XMP-drone-dji:AbsoluteAltitude']}",
           f"-XMP-drone-dji:RelativeAltitude={metadata['XMP-drone-dji:RelativeAltitude']}",
           f"-XMP-drone-dji:FlightYawDegree={metadata['XMP-drone-dji:FlightYawDegree']}",
           f"-XMP-drone-dji:FlightPitchDegree={metadata['XMP-drone-dji:FlightPitchDegree']}",
           f"-XMP-drone-dji:FlightRollDegree={metadata['XMP-drone-dji:FlightRollDegree']}",
           f"{image_file}"]
    result = subprocess.run(cmd, capture_output=True, text=True)
    return result

In [ ]:
def get_frame_metadata(exif_video_data:Dict, frame_number:int):
    frame_data = {"XMP-drone-dji:GPSLatitude": exif_video_data.get(f"Doc{str(frame_number)}:GPSLatitude", None),
                  "XMP-drone-dji:GPSLongitude":exif_video_data.get(f"Doc{str(frame_number)}:GPSLongitude", None),
                  "XMP-drone-dji:AbsoluteAltitude":exif_video_data.get(f"Doc{str(frame_number)}:AbsoluteAltitude", None),
                  "XMP-drone-dji:RelativeAltitude":exif_video_data.get(f"Doc{str(frame_number)}:RelativeAltitude", None),
                  "XMP-drone-dji:FlightYawDegree":exif_video_data.get(f"Doc{str(frame_number)}:DroneYaw", None),
                  "XMP-drone-dji:FlightPitchDegree":exif_video_data.get(f"Doc{str(frame_number)}:DronePitch", None),
                  "XMP-drone-dji:FlightRollDegree":exif_video_data.get(f"Doc{str(frame_number)}:DroneRoll", None),
                  "XMP-drone-dji:GimbalYawDegree":exif_video_data.get(f"Doc{str(frame_number)}:GimbalYaw", None),
                  "XMP-drone-dji:GimbalPitchDegree":exif_video_data.get(f"Doc{str(frame_number)}:GimbalPitch", None),
                  "XMP-drone-dji:GimbalRollDegree":exif_video_data.get(f"Doc{str(frame_number)}:GimbalRoll", None)}
    return frame_data


In [ ]:
def get_distance_between_camera_centers(camera1_metadata, camera2_metadata):
    # first we get the Abslout XYZ camera center value to measure the distance
    X1, Y1, Z1 = ecef_from_gps(lat_deg=camera1_metadata["XMP-drone-dji:GPSLatitude"],
                               long_deg=camera1_metadata["XMP-drone-dji:GPSLongitude"],
                               alt_m=camera1_metadata["XMP-drone-dji:AbsoluteAltitude"])
    
    X2, Y2, Z2 = ecef_from_gps(lat_deg=camera2_metadata["XMP-drone-dji:GPSLatitude"],
                               long_deg=camera2_metadata["XMP-drone-dji:GPSLongitude"],
                               alt_m=camera2_metadata["XMP-drone-dji:AbsoluteAltitude"])
    
    d = np.sqrt((X2 - X1)**2 + (Y2 - Y1)**2 + (Z2 - Z1)**2)

    return d

In [ ]:
def get_rotation_diff_between_cameras(camera1_metadata, camera2_metadata):
    yaw_diff = np.abs(camera2_metadata["XMP-drone-dji:GimbalYawDegree"] - camera1_metadata["XMP-drone-dji:GimbalYawDegree"])
    pitch_diff = np.abs(camera2_metadata["XMP-drone-dji:GimbalPitchDegree"] - camera1_metadata["XMP-drone-dji:GimbalPitchDegree"])
    roll_diff = np.abs(camera2_metadata["XMP-drone-dji:GimbalRollDegree"] - camera1_metadata["XMP-drone-dji:GimbalRollDegree"])

    return yaw_diff, pitch_diff, roll_diff

In [ ]:
def process_video_to_images(video_file, image_dir, 
                            frame_step=1, max_frame_number=None, use_constant_step=False, 
                            use_camera_movement=True, min_camera_distanct_m=0.3, min_camera_rot_deg=10, 
                            use_prev_if_value_missing=False, assume_missing_zero=True,
                            write_standard_gps=True, debug_prints=True):
    os.makedirs(image_dir, exist_ok=True)

    cap = cv2.VideoCapture(video_file)
    frame_index = 0
    video_metadata =  read_video_metadata_exiftool(video_file)
    prev_frame_metadata = None
    prev_saved_frame_metadata = None
    missing_values = 0
    

    while True:

        ret, frame = cap.read()
        save_frame = False

        if not ret:
            break

        if max_frame_number is not None:
            if frame_index >= max_frame_number:
                break
        
        # The meta data frame number starts from 1
        frame_metadata = get_frame_metadata(video_metadata, frame_number=frame_index+1)
        
        for key, value in frame_metadata.items():
            if value is None:
                if assume_missing_zero:
                    frame_metadata[key] = 0
                    if debug_prints:
                        print(f"Warning: frame: {frame_index} missing value: {key} assumed zero")
                elif prev_frame_metadata is not None and use_prev_if_value_missing and prev_frame_metadata[key] is not None:
                    frame_metadata[key] = prev_frame_metadata[key]
                    if debug_prints:
                        print(f"Warning: frame: {frame_index} missing value: {key} previous value used")
                missing_values += 1
        
        if use_constant_step and frame_index % frame_step == 0:
            save_frame = True
        elif use_camera_movement:
            if prev_saved_frame_metadata is None:
                # this is the frame so we will save it anyway
                save_frame = True
            else:
                distance = get_distance_between_camera_centers(prev_saved_frame_metadata, frame_metadata)
                yaw_diff, pitch_diff, roll_diff = get_rotation_diff_between_cameras(prev_saved_frame_metadata, frame_metadata)

                if distance >= min_camera_distanct_m or yaw_diff >= min_camera_rot_deg or \
                   pitch_diff >= min_camera_rot_deg or roll_diff >= min_camera_rot_deg:
                    save_frame = True
                    if debug_prints:
                        print(f"Saving frame with camera dist={distance}, yaw_diff={yaw_diff}, roll_diff={roll_diff}, pitch_diff={pitch_diff}")

        # We read all the meta data to keep track of missing params but only write when needed
        if save_frame:
            frame_filename = f"{image_dir}/frame_{frame_index:06d}.JPG"
            cv2.imwrite(frame_filename, frame)
            # Write the gimble meta data
            write_all_drone_tags(frame_metadata, frame_filename)
            prev_saved_frame_metadata = frame_metadata
            
            if write_standard_gps:
                write_standard_gps_tags(frame_metadata, frame_filename)

        prev_frame_metadata = frame_metadata

        frame_index += 1

    cap.release()

    # return the number of frames in the video and the number of missing data
    return frame_index+1, missing_values

## Test Code

In [ ]:
root_dir = "G:/Mary/Picture/drone/celitic_cross_full_files"
video_file = f"{root_dir}/DJI_20250504165937_0075_D.MP4"
image_dir = f"{root_dir}/images"


In [ ]:
exif_video_data = read_video_metadata_exiftool(video_file)

In [ ]:
frames = 300

for frame in range(frames):
    data = get_frame_metadata(exif_video_data, frame+1)
    print(f"frame: {frame} gimble yaw: {data['XMP-drone-dji:GimbalYawDegree']}")
    print(f"frame: {frame} flight yaw: {data['XMP-drone-dji:FlightYawDegree']}")

In [ ]:
frames = 100

for frame in range(frames):
    data = get_frame_metadata(exif_video_data, frame+1)
    print(f"frame: {frame} gimble pitch: {data['XMP-drone-dji:GimbalPitchDegree']}")
    print(f"frame: {frame} flight pitch: {data['XMP-drone-dji:FlightPitchDegree']}")

In [ ]:
print(json.dumps(exif_video_data, indent=4))

In [ ]:
frame_number = 1
frame_data = get_frame_metadata(exif_video_data, frame_number)
print(json.dumps(frame_data, indent=4))

In [ ]:
frame_step = 2
max_frame_number = None
use_prev_if_value_missing = False
debug = True
assume_missing_zero = True
write_standard_gps = True
use_camera_differance = True
min_distance_m = 0.3
min_rot_degree = 10

number_of_frames, missing_values = process_video_to_images(video_file=video_file, image_dir=image_dir,
                                                           frame_step=frame_step, max_frame_number=max_frame_number, use_camera_movement=use_camera_differance,
                                                           min_camera_distanct_m=min_distance_m, min_camera_rot_deg=min_rot_degree,
                                                           use_prev_if_value_missing=use_prev_if_value_missing, assume_missing_zero=assume_missing_zero,
                                                           write_standard_gps=write_standard_gps, debug_prints=debug)



In [ ]:
print(f"total number of frames = {number_of_frames}, missing data = {missing_values}")